# Spectra pooling: mean vs DeepSets vs attention (prototype, Member 7)

**Question:** a test molecule has 1–16 spectra (median 3). Does a *learned* way of combining them
beat simply averaging the per-spectrum predictions?

**Pipeline (one Run all on a Colab GPU):**
1. Load a subset of `train.parquet`, hold out 10% of structures (by `inchikey14`) for validation.
2. Clean spectra the way the host cleaned the test set; bin them (2200 × 1 Da, sqrt), add metadata.
3. **Stage A** – train one per-spectrum encoder: spectrum → 1024-bit Morgan fingerprint.
4. **Stage B** – on the *same frozen encoder*, compare three ways to pool a molecule's spectra:
   `mean` (no training, baseline) · `deepsets` (arch2) · `attention` (arch1).
5. Rank mass-matched candidate structures by Tanimoto to the pooled prediction; report MRR@25.

**Stand-ins (replace when teammates' pieces land):**

| Piece | Stand-in here | Real owner |
|---|---|---|
| Validation split | random 10% of `inchikey14`, seed 2026 | Member 5 (`data/splits/`) |
| Candidates | all training structures within 10 ppm of the **true** neutral mass (so the answer is always in the list: optimistic) | Member 4 (`casmi.candidates`) |
| Scorer | `common/evaluate.py` MRR@25 on inchikey14 | Member 5 (`casmi.evaluation`) |
| Fingerprint model | small MLP encoder trained here | Member 1 / Member 2 |

Absolute MRR is therefore **not** comparable to the leaderboard; the comparison *between pooling
methods* on identical inputs is the result.

**Outputs:** small results (CSV/JSON/PNG) → `Project/outputs/pooling_v1/` (synced to the Mac);
heavy files (data copy, caches, checkpoints, score parquets) → `MyDrive/DATA266_CASMI/`.

In [1]:
# ---- Config (env vars override, used for the local smoke test) ----
import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
E = os.environ.get

MAC = "/content/drive/Othercomputers/My Mac/data266-9067"
REPO = E("POOL_REPO", f"{MAC}/Project/01_repo/Team-10-Enveda-CASMI-2026")
RESULTS_DIR = E("POOL_RESULTS_DIR", f"{MAC}/Project/outputs/pooling_v1")        # small, synced
HEAVY_DIR = E("POOL_HEAVY_DIR", "/content/drive/MyDrive/DATA266_CASMI/pooling_v1")  # big, Drive
DRIVE_DATA = E("POOL_DRIVE_DATA", "/content/drive/MyDrive/DATA266_CASMI/data")    # train.parquet lives here
TRAIN_PARQUET = E("POOL_TRAIN_PARQUET", "")   # set to skip the Drive/Kaggle lookup

SEED = 2026
VAL_FRAC = 0.10                                   # matches src/casmi/evaluation/split.py
N_TRAIN_MOL = int(E("POOL_N_TRAIN_MOL", 50000))   # training structures used (of ~248k)
N_VAL_MOL = int(E("POOL_N_VAL_MOL", 3000))        # validation molecules scored
MAX_SPEC_PER_MOL = 16
FP_SIZE, FP_RADIUS = 1024, 2                      # same as Member 2's CNN
MZ_MAX, BIN_WIDTH = 2200, 1.0                     # same as Member 2's CNN
PPM = 10.0                                        # candidate mass window
ENC_EPOCHS = int(E("POOL_ENC_EPOCHS", 8))
POOL_EPOCHS = int(E("POOL_POOL_EPOCHS", 15))
BATCH, SET_BATCH = 512, 256
AUTO_DISCONNECT = E("POOL_AUTO_DISCONNECT", "1") == "1"
print("Colab:", IN_COLAB, "| train molecules:", N_TRAIN_MOL, "| val molecules:", N_VAL_MOL)

Colab: True | train molecules: 50000 | val molecules: 3000


In [2]:
# ---- Setup: Drive, imports, release the GPU on any error ----
import json
import subprocess
import sys
import time

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    def _release(shell, etype, evalue, tb, tb_offset=None):
        shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
        if AUTO_DISCONNECT:
            print("Error: releasing the GPU runtime.")
            from google.colab import runtime
            runtime.unassign()
    get_ipython().set_custom_exc((Exception,), _release)
try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit==2026.3.3"], check=True)
    import rdkit
assert rdkit.__version__ == "2026.03.3", rdkit.__version__
for p in (f"{REPO}/src", f"{REPO}/models/spectra_pooling"):
    if p not in sys.path:
        sys.path.insert(0, p)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(HEAVY_DIR, exist_ok=True)

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")
from common.evaluate import ranks_of_truth, score_queries, summarise
from common.pooling import SpectrumEncoder, build_pool, mean_probabilities
from common.sets import group_indices, pad_sets, sample_sets

from casmi.interfaces import write_scores
from casmi.preprocess.binning import bin_spectrum

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")
T0 = time.time()

Mounted at /content/drive
device: cuda Tesla T4


In [3]:
# ---- Data: train.parquet from Drive (or Kaggle API once), copied to local disk for speed ----
import shutil

if not TRAIN_PARQUET:
    src = f"{DRIVE_DATA}/train.parquet"
    if not os.path.exists(src):
        # One-time download. Needs the competition rules accepted on your Kaggle account and
        # Colab secrets: either KAGGLE_API_TOKEN (new single token), or KAGGLE_USERNAME + KAGGLE_KEY.
        from google.colab import userdata
        def _secret(name):
            try:
                return userdata.get(name)
            except Exception:
                return None
        token = _secret("KAGGLE_API_TOKEN")
        if token:
            os.environ["KAGGLE_API_TOKEN"] = token
        else:
            os.environ["KAGGLE_USERNAME"] = _secret("KAGGLE_USERNAME") or ""
            os.environ["KAGGLE_KEY"] = _secret("KAGGLE_KEY") or ""
            if not os.environ["KAGGLE_KEY"]:
                raise RuntimeError("Add Colab secret KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY), "
                                   "or put train.parquet in DRIVE_DATA.")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle>=1.7.4"], check=True)
        os.makedirs(DRIVE_DATA, exist_ok=True)
        dl = subprocess.run(["kaggle", "competitions", "download", "-c",
                             "enveda-CASMI26-molecule-id-mass-spectra", "-p", "/content/raw"],
                            capture_output=True, text=True)
        print(dl.stdout[-2000:], dl.stderr[-2000:])
        if dl.returncode:
            raise RuntimeError("Kaggle download failed (message above). 401 = token not read or wrong; "
                               "403 = accept the competition rules on Kaggle first.")
        subprocess.run(["unzip", "-o", "-q", "/content/raw/enveda-CASMI26-molecule-id-mass-spectra.zip",
                        "-d", "/content/raw"], check=True)
        shutil.copy("/content/raw/train.parquet", src)
    TRAIN_PARQUET = "/content/train.parquet"
    if not os.path.exists(TRAIN_PARQUET):
        shutil.copy(src, TRAIN_PARQUET)
pf = pq.ParquetFile(TRAIN_PARQUET)
print(f"{pf.metadata.num_rows:,} rows in train.parquet")


  [00:14<00:04, 254MB/s]
 65%|██████▍   | 1.83G/2.82G [00:15<00:04, 249MB/s]
 66%|██████▌   | 1.85G/2.82G [00:15<00:04, 249MB/s]
 66%|██████▋   | 1.87G/2.82G [00:15<00:04, 242MB/s]
 67%|██████▋   | 1.90G/2.82G [00:15<00:04, 243MB/s]
 68%|██████▊   | 1.92G/2.82G [00:15<00:04, 234MB/s]
 69%|██████▉   | 1.95G/2.82G [00:15<00:03, 241MB/s]
 70%|██████▉   | 1.97G/2.82G [00:15<00:03, 236MB/s]
 71%|███████   | 1.99G/2.82G [00:15<00:03, 245MB/s]
 72%|███████▏  | 2.02G/2.82G [00:15<00:03, 268MB/s]
 73%|███████▎  | 2.05G/2.82G [00:16<00:02, 284MB/s]
 74%|███████▎  | 2.08G/2.82G [00:16<00:02, 275MB/s]
 75%|███████▍  | 2.11G/2.82G [00:16<00:02, 279MB/s]
 76%|███████▌  | 2.14G/2.82G [00:16<00:02, 281MB/s]
 77%|███████▋  | 2.16G/2.82G [00:16<00:02, 278MB/s]
 78%|███████▊  | 2.19G/2.82G [00:16<00:02, 272MB/s]
 78%|███████▊  | 2.21G/2.82G [00:16<00:02, 267MB/s]
 79%|███████▉  | 2.24G/2.82G [00:16<00:02, 265MB/s]
 80%|████████  | 2.27G/2.82G [00:16<00:02, 259MB/s]
 81%|████████  | 2.29G/2.82G [00:16<00

In [4]:
# ---- Structure-level split (stand-in for Member 5) and molecule subset ----
labels = pq.read_table(TRAIN_PARQUET, columns=["inchikey14", "normalized_smiles"]).to_pandas()
structures = labels.drop_duplicates("inchikey14").set_index("inchikey14")["normalized_smiles"]
keys = np.array(sorted(structures.index))
perm = np.random.default_rng(SEED).permutation(len(keys))
n_val_all = int(round(VAL_FRAC * len(keys)))
val_all, train_all = keys[perm[:n_val_all]], keys[perm[n_val_all:]]
train_keys = rng.choice(train_all, size=min(N_TRAIN_MOL, len(train_all)), replace=False)
val_keys = rng.choice(val_all, size=min(N_VAL_MOL, len(val_all)), replace=False)
assert not set(train_keys) & set(val_all), "a validation structure leaked into training"
print(f"{len(keys):,} structures | split {len(train_all):,} train / {len(val_all):,} val "
      f"| using {len(train_keys):,} train / {len(val_keys):,} val")
del labels

275,810 structures | split 248,229 train / 27,581 val | using 50,000 train / 3,000 val


In [5]:
# ---- Stream the selected molecules' spectra and clean them like the test set ----
COLS = ["inchikey14", "ms2_mzs", "ms2_normalized_intensities", "precursor_mz", "base_peak_intensity",
        "collision_energy_ev", "adduct", "ionization_mode", "instrument_type", "ingest_lib"]
wanted = pa.array(np.concatenate([train_keys, val_keys]))
parts = []
for batch in pf.iter_batches(columns=COLS, batch_size=100_000):
    tbl = pa.Table.from_batches([batch])
    tbl = tbl.filter(pc.is_in(tbl["inchikey14"], value_set=wanted))
    if tbl.num_rows:
        parts.append(tbl.to_pandas())
df = pd.concat(parts, ignore_index=True)
del parts
n0 = len(df)
df = df[~(df.base_peak_intensity.notna() & (df.base_peak_intensity < 1000))]   # host: base peak < 1,000 dropped
df = df.sample(frac=1.0, random_state=SEED)                                       # random ≤16 spectra per molecule
df = df[df.groupby("inchikey14").cumcount() < MAX_SPEC_PER_MOL].reset_index(drop=True)
def trim(row):  # host: peaks > precursor + 2 Da removed
    mz, it = np.asarray(row.ms2_mzs, float), np.asarray(row.ms2_normalized_intensities, float)
    keep = mz <= row.precursor_mz + 2.0
    return mz[keep], it[keep]
trimmed = [trim(r) for r in df.itertuples()]
df["mzs"] = [t[0] for t in trimmed]
df["ints"] = [t[1] for t in trimmed]
df = df[df.mzs.map(len) > 0].reset_index(drop=True)
df["split"] = np.where(df.inchikey14.isin(set(val_keys)), "val", "train")
print(f"{n0:,} rows read → {len(df):,} after cleaning and ≤{MAX_SPEC_PER_MOL}/molecule")
print(df.split.value_counts().to_dict())

482,408 rows read → 356,104 after cleaning and ≤16/molecule
{'train': 335605, 'val': 20499}


In [6]:
# ---- Features: binned spectrum (sqrt) + metadata ----
TEST_ADDUCTS = ["[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
                "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-"]
n_bins = int(np.ceil(MZ_MAX / BIN_WIDTH))
X = np.zeros((len(df), n_bins), dtype=np.float16)
for i, (mz, it) in enumerate(zip(df.mzs, df.ints, strict=False)):
    X[i] = bin_spectrum(mz, it, max_mz=MZ_MAX, bin_width=BIN_WIDTH, sqrt_intensity=True)
def ce_mean(v):
    try:
        v = np.asarray(v, float)
        return float(v.mean()) if v.size else np.nan
    except (TypeError, ValueError):
        return np.nan
ce = df.collision_energy_ev.map(ce_mean).to_numpy()
meta = np.column_stack([
    np.nan_to_num(ce / 100.0), np.isnan(ce),                                   # collision energy + missing flag
    (df.ionization_mode == "positive"),
    df.instrument_type.fillna("").str.contains("tims", case=False),             # same platform as test
    *[(df.adduct == a) for a in TEST_ADDUCTS], ~df.adduct.isin(TEST_ADDUCTS),
]).astype(np.float32)
mol_keys = np.array(sorted(set(df.inchikey14)))
mol_index = {k: i for i, k in enumerate(mol_keys)}
row_mol = df.inchikey14.map(mol_index).to_numpy()
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE, includeChirality=False)
def fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return None if mol is None else fpgen.GetFingerprintAsNumPy(mol).astype(np.uint8)
FP = {k: fingerprint(structures[k]) for k in mol_keys}
bad = [k for k, v in FP.items() if v is None]
assert len(bad) < 0.01 * len(FP), f"{len(bad)} unparseable SMILES"
for k in bad:
    FP[k] = np.zeros(FP_SIZE, np.uint8)
Y = np.stack([FP[k] for k in mol_keys])
is_val = (df.split == "val").to_numpy()
print("X", X.shape, "meta", meta.shape, "Y", Y.shape, f"| {time.time()-T0:.0f}s")

X (356104, 2200) meta (356104, 15) Y (52798, 1024) | 199s


In [7]:
# ---- Stage A: per-spectrum encoder (spectrum -> fingerprint), shared by all three poolers ----
Xt = torch.from_numpy(X).to(DEV)
Mt = torch.from_numpy(meta).to(DEV)
Yt = torch.from_numpy(Y).to(DEV).float()
Rt = torch.from_numpy(row_mol).to(DEV)
tr_rows = torch.from_numpy(np.flatnonzero(~is_val)).to(DEV)
va_rows = torch.from_numpy(np.flatnonzero(is_val)).to(DEV)
enc = SpectrumEncoder(n_bins, meta.shape[1], FP_SIZE).to(DEV)
opt = torch.optim.AdamW(enc.parameters(), lr=1e-3, weight_decay=1e-4)
def enc_loss(rows):
    return F.binary_cross_entropy_with_logits(enc(Xt[rows].float(), Mt[rows]), Yt[Rt[rows]])
best, hist = float("inf"), []
for ep in range(ENC_EPOCHS):
    enc.train()
    for b in tr_rows[torch.randperm(len(tr_rows), device=DEV)].split(BATCH):
        opt.zero_grad()
        loss = enc_loss(b)
        loss.backward()
        opt.step()
    enc.eval()
    with torch.no_grad():
        vl = float(torch.stack([enc_loss(b) * len(b) for b in va_rows.split(4096)]).sum() / len(va_rows))
    hist.append({"stage": "encoder", "epoch": ep + 1, "val_bce": vl})
    if vl < best:
        best = vl
        torch.save(enc.state_dict(), f"{HEAVY_DIR}/encoder.pt")
    print(f"encoder epoch {ep+1}: val BCE {vl:.4f}")
enc.load_state_dict(torch.load(f"{HEAVY_DIR}/encoder.pt", map_location=DEV))
enc.eval()
with torch.no_grad():
    H = torch.cat([enc.embed(Xt[i].float(), Mt[i]) for i in torch.arange(len(df), device=DEV).split(4096)])
    P = torch.sigmoid(enc.head(H))
Xt = None  # free GPU memory
print("embeddings", tuple(H.shape), f"| {time.time()-T0:.0f}s")

encoder epoch 1: val BCE 0.1367
encoder epoch 2: val BCE 0.1345
encoder epoch 3: val BCE 0.1332
encoder epoch 4: val BCE 0.1323
encoder epoch 5: val BCE 0.1316
encoder epoch 6: val BCE 0.1310
encoder epoch 7: val BCE 0.1307
encoder epoch 8: val BCE 0.1304
embeddings (356104, 512) | 224s


In [8]:
# ---- Fixed validation queries: each val molecule gets a test-like set of 1-16 of its spectra ----
groups = group_indices(df.inchikey14.to_numpy())
val_mols = [k for k in mol_keys if k in set(val_keys)]
val_sets = sample_sets(groups, val_mols, np.random.default_rng(7))
query_ids = [f"val_{i:05d}" for i in range(len(val_mols))]
truth = dict(zip(query_ids, val_mols, strict=False))
set_size = np.array([len(s) for s in val_sets])
print(f"{len(val_sets):,} val queries | spectra per query: median {np.median(set_size):.0f}, "
      f"max {set_size.max()} | {np.bincount(set_size)[1:6].tolist()} with 1..5")
train_mols = [k for k in mol_keys if k not in set(val_keys)]

2,986 val queries | spectra per query: median 3, max 16 | [874, 611, 530, 430, 158] with 1..5


In [9]:
# ---- Stage B: train DeepSets and attention pooling on the frozen encoder ----
def pooled_targets(mols):
    return torch.from_numpy(np.stack([FP[k] for k in mols])).to(DEV).float()
Yval = pooled_targets(val_mols)
def run_pool(pool, sets):
    return pool(*pad_sets(sets, H))
pools, att_val = {}, None
for name in ["deepsets", "attention"]:
    torch.manual_seed(SEED)
    pool = build_pool(name, H.shape[1], FP_SIZE).to(DEV)
    opt = torch.optim.AdamW(pool.parameters(), lr=1e-3, weight_decay=1e-4)
    best = float("inf")
    prng = np.random.default_rng(SEED)
    for ep in range(POOL_EPOCHS):
        pool.train()
        order = prng.permutation(len(train_mols))
        for s in range(0, len(order), SET_BATCH):
            mols = [train_mols[j] for j in order[s:s + SET_BATCH]]
            logits, _ = run_pool(pool, sample_sets(groups, mols, prng))
            loss = F.binary_cross_entropy_with_logits(logits, pooled_targets(mols))
            opt.zero_grad()
            loss.backward()
            opt.step()
        pool.eval()
        with torch.no_grad():
            vl = float(sum(F.binary_cross_entropy_with_logits(run_pool(pool, val_sets[s:s+1024])[0],
                           Yval[s:s+1024], reduction="sum") for s in range(0, len(val_sets), 1024))
                       / (len(val_sets) * FP_SIZE))
        hist.append({"stage": name, "epoch": ep + 1, "val_bce": vl})
        if vl < best:
            best = vl
            torch.save(pool.state_dict(), f"{HEAVY_DIR}/{name}.pt")
        print(f"{name} epoch {ep+1}: val BCE {vl:.4f}")
    pool.load_state_dict(torch.load(f"{HEAVY_DIR}/{name}.pt", map_location=DEV))
    pool.eval()
    pools[name] = pool
print(f"| {time.time()-T0:.0f}s")

deepsets epoch 1: val BCE 0.1366
deepsets epoch 2: val BCE 0.1340
deepsets epoch 3: val BCE 0.1318
deepsets epoch 4: val BCE 0.1304
deepsets epoch 5: val BCE 0.1293
deepsets epoch 6: val BCE 0.1286
deepsets epoch 7: val BCE 0.1280
deepsets epoch 8: val BCE 0.1277
deepsets epoch 9: val BCE 0.1276
deepsets epoch 10: val BCE 0.1270
deepsets epoch 11: val BCE 0.1268
deepsets epoch 12: val BCE 0.1267
deepsets epoch 13: val BCE 0.1265
deepsets epoch 14: val BCE 0.1262
deepsets epoch 15: val BCE 0.1260
attention epoch 1: val BCE 0.1366
attention epoch 2: val BCE 0.1343
attention epoch 3: val BCE 0.1315
attention epoch 4: val BCE 0.1300
attention epoch 5: val BCE 0.1293
attention epoch 6: val BCE 0.1284
attention epoch 7: val BCE 0.1279
attention epoch 8: val BCE 0.1276
attention epoch 9: val BCE 0.1273
attention epoch 10: val BCE 0.1266
attention epoch 11: val BCE 0.1266
attention epoch 12: val BCE 0.1264
attention epoch 13: val BCE 0.1261
attention epoch 14: val BCE 0.1259
attention epoch 15

In [10]:
# ---- Pooled predictions for every val query ----
preds = {}
with torch.no_grad():
    preds["mean"] = torch.cat([mean_probabilities(*pad_sets(val_sets[s:s+1024], P))
                               for s in range(0, len(val_sets), 1024)]).cpu().numpy()
    for name, pool in pools.items():
        out, weights = [], []
        for s in range(0, len(val_sets), 1024):
            logits, w = run_pool(pool, val_sets[s:s+1024])
            out.append(torch.sigmoid(logits).cpu())
            if w is not None:
                weights += [w[i, :len(val_sets[s+i])].cpu().numpy() for i in range(len(w))]
        preds[name] = torch.cat(out).numpy()
        if name == "attention":
            att_val = weights
print({k: v.shape for k, v in preds.items()})

{'mean': (2986, 1024), 'deepsets': (2986, 1024), 'attention': (2986, 1024)}


In [11]:
# ---- Candidates (stand-in for Member 4): structures within PPM of the true neutral mass ----
def exact_mass(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return np.nan if mol is None else Descriptors.ExactMolWt(mol)
mass_cache = f"{HEAVY_DIR}/structure_masses.parquet"
if os.path.exists(mass_cache) and pd.read_parquet(mass_cache).shape[0] == len(structures):
    masses = pd.read_parquet(mass_cache)["mass"]
else:
    masses = pd.Series([exact_mass(s) for s in structures.to_numpy()], index=structures.index, name="mass")
    masses.to_frame().to_parquet(mass_cache)
masses = masses.dropna().sort_values()
m_keys, m_vals = masses.index.to_numpy(), masses.to_numpy()
candidates = {}
for qid, key in truth.items():
    m = masses.get(key, np.nan)
    if np.isnan(m):
        candidates[qid] = [key]
        continue
    lo, hi = np.searchsorted(m_vals, [m * (1 - PPM * 1e-6), m * (1 + PPM * 1e-6)])
    candidates[qid] = list(m_keys[lo:hi]) or [key]
for k in {k for c in candidates.values() for k in c} - set(FP):
    fp = fingerprint(structures[k])
    FP[k] = fp if fp is not None else np.zeros(FP_SIZE, np.uint8)
n_cand = np.array([len(c) for c in candidates.values()])
print(f"candidates per query: median {np.median(n_cand):.0f}, mean {n_cand.mean():.0f}, max {n_cand.max()} "
      f"| {np.mean(n_cand == 1):.0%} have a single candidate (trivially rank 1)")

candidates per query: median 57, mean 87, max 500 | 3% have a single candidate (trivially rank 1)


In [12]:
# ---- Score, rank and compare ----
MODEL_IDS = {"mean": "spectra_pooling.mean_baseline.prototype",
             "deepsets": "spectra_pooling.arch2_deepsets.prototype",
             "attention": "spectra_pooling.arch1_attention_pooling.prototype"}
rows, ranks = [], {}
size_bucket = pd.cut(set_size, [0, 1, 3, 7, 16], labels=["1", "2-3", "4-7", "8-16"])
hard = n_cand > 1
for name, pr in preds.items():
    scores = score_queries(pr, query_ids, candidates, FP)
    write_scores(scores, MODEL_IDS[name], "val", root=HEAVY_DIR)
    r = ranks_of_truth(scores, truth)
    ranks[name] = r
    rows.append({"method": name, "subset": "all", **summarise(r)})
    rows.append({"method": name, "subset": ">1 candidate", **summarise(r[hard])})
    for b in size_bucket.categories:
        rows.append({"method": name, "subset": f"{b} spectra", **summarise(r[np.asarray(size_bucket == b)])})
results = pd.DataFrame(rows)
pd.set_option("display.precision", 3)
display(results.pivot(index="subset", columns="method", values="mrr@25")
        .reindex(["all", ">1 candidate", "1 spectra", "2-3 spectra", "4-7 spectra", "8-16 spectra"]))
display(results[results.subset.isin(["all", ">1 candidate"])])

method,attention,deepsets,mean
subset,,,
all,0.338,0.332,0.310
>1 candidate,0.321,0.315,0.292
1 spectra,0.346,0.341,0.328
2-3 spectra,0.341,0.338,0.315
4-7 spectra,0.318,0.313,0.284
8-16 spectra,0.372,0.343,0.300


,method,subset,mrr@25,top1,top5,top25,n
0,mean,all,0.310,0.185,0.448,0.751,2986
1,mean,>1 candidate,0.292,0.163,0.434,0.745,2910
6,deepsets,all,0.332,0.203,0.478,0.774,2986
7,deepsets,>1 candidate,0.315,0.182,0.465,0.768,2910
12,attention,all,0.338,0.209,0.485,0.782,2986
13,attention,>1 candidate,0.321,0.188,0.471,0.777,2910


In [13]:
# ---- Does attention find the better spectra? (only queries with >= 2 spectra) ----
def tanimoto01(p, y):
    inter = (p * y).sum()
    return inter / max(p.sum() + y.sum() - inter, 1e-8)
Pn = P.cpu().numpy()
picked, chance = [], []
for s, w, key in zip(val_sets, att_val, val_mols, strict=False):
    if len(s) < 2:
        continue
    q = np.array([tanimoto01(Pn[i], FP[key]) for i in s])   # how good each spectrum is on its own
    picked.append(q[np.argmax(w)])
    chance.append(q.mean())
picked, chance = np.array(picked), np.array(chance)
att_check = {"queries": int(len(picked)),
             "top_attention_spectrum_quality": float(picked.mean()),
             "average_spectrum_quality": float(chance.mean()),
             "share_top_attention_beats_average": float(np.mean(picked > chance))}
print(json.dumps(att_check, indent=2))

{
  "queries": 2112,
  "top_attention_spectrum_quality": 0.18727537549223558,
  "average_spectrum_quality": 0.17811428415747774,
  "share_top_attention_beats_average": 0.7618371212121212
}


In [14]:
# ---- Save small results to the synced folder; heavy files are already on Drive ----
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

results.to_csv(f"{RESULTS_DIR}/pooling_results.csv", index=False)
pd.DataFrame(hist).to_csv(f"{RESULTS_DIR}/training_history.csv", index=False)
summary = {"date": time.strftime("%Y-%m-%d %H:%M"), "runtime_min": round((time.time() - T0) / 60, 1),
           "device": str(DEV), "n_train_mol": len(train_mols), "n_val_queries": len(val_sets),
           "n_spectra": int(len(df)), "candidates_median": float(np.median(n_cand)),
           "config": {"fp": [FP_SIZE, FP_RADIUS], "bins": [MZ_MAX, BIN_WIDTH], "ppm": PPM,
                      "enc_epochs": ENC_EPOCHS, "pool_epochs": POOL_EPOCHS, "seed": SEED},
           "mrr@25": {m: float(v) for m, v in
                      results[results.subset == "all"].set_index("method")["mrr@25"].items()},
           "attention_check": att_check,
           "stand_ins": "own 10% split, oracle-mass candidates from training structures, own scorer"}
with open(f"{RESULTS_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
sub = results[results.subset.str.endswith("spectra")]
ax = sub.pivot(index="subset", columns="method", values="mrr@25").reindex(
    ["1 spectra", "2-3 spectra", "4-7 spectra", "8-16 spectra"]).plot.bar(figsize=(7, 4), rot=0)
ax.set_ylabel("MRR@25 (stand-in candidates)")
ax.set_xlabel("spectra per molecule")
ax.set_title("Pooling method vs. number of spectra")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/mrr_by_set_size.png", dpi=150)
plt.close()
print(json.dumps(summary["mrr@25"], indent=2))
print("saved to", RESULTS_DIR)

{
  "mean": 0.3096432511171493,
  "deepsets": 0.33219330172101524,
  "attention": 0.3379621823081315
}
saved to /content/drive/Othercomputers/My Mac/data266-9067/Project/outputs/pooling_v1


In [15]:
# ---- Done: release the GPU ----
print(f"total {(time.time()-T0)/60:.1f} min")
if IN_COLAB and AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()

total 7.9 min
